<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I’m going with Logistic Regression for the first pass. Clustering doesn't make sense since we have labels, and I want to keep things interpretable before jumping into tree ensembles. I want to see if the baseline features actually hold up in a linear model before adding complexity. If LR can't beat the baseline, it's probably a feature engineering issue, not a model issue.

**Core Hypotheses**


- **H1 (Missing Signal Hypothesis):** Pages with `avg_position == 0` represent unranked pages rather than genuine ranking positions. Adding an `avg_position_missing` indicator should help Logistic Regression separate these cases and improve Precision@50.

- **H2 (Traffic Skew Hypothesis):** `impressions_90d` is heavily right-skewed. Applying `log1p(impressions_90d)` should reduce the influence of extreme values and improve the stability of the linear model.

- **H3 (Non-Linear Interaction Hypothesis):** Logistic Regression cannot naturally learn threshold rules such as "only trust CTR when impressions are sufficiently high." A Random Forest should capture these interactions and outperform both Logistic Regression and the rule-based baseline.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used **GroupKFold (5 folds)** grouped by `client_id`.

A random train/test split can leak information because pages from the same client often share similar SEO behaviour. Keeping entire clients within a single fold produces a more realistic estimate of how the model performs on unseen client portfolios.

Before training, I also compared several `GroupShuffleSplit` seeds to confirm that random client assignment caused noticeable variation in Precision@50, reinforcing the decision to use grouped cross-validation account.

In [1]:
#All imports
import os
import subprocess
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from IPython.display import display


In [8]:

%pip install -q duckdb huggingface_hub
%pip install -q duckdb

In [7]:
# Step 1 — Fetch data from starter repo

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    # wildcard + hive_partitioning=true grabs every month and exposes it as a 'month' column
    "fact_daily_all": f"read_parquet('{REL}/fact_content_daily_performance/month=*/*.parquet', hive_partitioning=true)",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:20} {n:>12,} rows")

# check exactly which months you actually have before deciding your feature/outcome window
con.sql(f"SELECT month, COUNT(*) FROM {TABLES['fact_daily_all']} GROUP BY month ORDER BY month").show()

dim_clients                   104 rows
dim_content               519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily_all         78,835,655 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬──────────────┐
│  month  │ count_star() │
│ varchar │    int64     │
├─────────┼──────────────┤
│ 2025-01 │         1297 │
│ 2025-02 │        75985 │
│ 2025-03 │       167859 │
│ 2025-04 │       285114 │
│ 2025-05 │       349923 │
│ 2025-06 │       329201 │
│ 2025-07 │       469794 │
│ 2025-08 │       704962 │
│ 2025-09 │       845813 │
│ 2025-10 │      2165471 │
│ 2025-11 │      6793825 │
│ 2025-12 │      7752930 │
│ 2026-01 │      7890817 │
│ 2026-02 │      7355108 │
│ 2026-03 │      9841378 │
│ 2026-04 │     10424730 │
│ 2026-05 │     11687376 │
│ 2026-06 │     11694072 │
├─────────┴──────────────┤
│ 18 rows      2 columns │
└────────────────────────┘



In [ ]:
df = con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily_all']} f
    LEFT JOIN {TABLES['dim_content']} c USING (content_id)
    LEFT JOIN {TABLES['dim_clients']} cl USING (client_id)
""").df()

print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
# Step 2 —  Basic dataset checks
print(f"Dataset shape: {df.shape}")
print(f"Duplicate content_ids: {df['content_id'].duplicated().sum()}")
print(f"Overall decline rate: {df['trend_direction'].eq('down').mean():.3f}")
print("duplicate content_id rows:", df["content_id"].duplicated().sum())

In [ ]:
# Step 3 —  Create target label and rule-based indicators
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

STALE_TIERS = ["91-180", "181+"]
CTR_THRESHOLD = 0.005
IMPRESSION_DECOY_LEVEL = 5000

df["stale_flag"] = df["freshness_tier"].isin(STALE_TIERS)
df["low_ctr_flag"] = (df["avg_position"] <= 20) & (df["ctr"] < CTR_THRESHOLD) & (df["impressions_90d"] >= 500)
df["is_decoy"] = (df["freshness_tier"] == "181+") & (df["impressions_90d"] >= IMPRESSION_DECOY_LEVEL)

def calculate_score(row):
    if row["is_decoy"]: return 3
    if row["stale_flag"] and row["low_ctr_flag"]: return 2
    if row["stale_flag"] or row["low_ctr_flag"]: return 1
    return 0

df["baseline_score"] = df.apply(calculate_score, axis=1)

First Attempt

In [ ]:
# Step 4 — Initial random split check
# This is exploratory only. Final evaluation uses GroupKFold by client.


train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["is_declining_label"]
)
print(train_df.shape, test_df.shape)



(24000, 49) (6000, 49)


In [ ]:
# Step 5 — exploratory baseline check
# Final baseline evaluation is reported using the same 5-fold GroupKFold
# as the machine-learning models.
def precision_at_k(sub_df, score_col, k=50, tiebreak_col="impressions_90d"):
    top_k = sub_df.sort_values([score_col, tiebreak_col], ascending=[False, False]).head(k)
    return top_k["is_declining_label"].mean()

baseline_p50 = precision_at_k(test_df, "baseline_score", k=50)
print(f"Baseline Precision@50 (test set only): {baseline_p50:.3f}")

Baseline Precision@50 (test set only): 0.880


In [ ]:
# Where did the biggest client end up in your current split?
top_client = df["client_id"].value_counts().idxmax()
print("biggest client:", top_client, "-> in test set:", top_client in test_df["client_id"].values)

# Check stability: with only 32 groups, one random split can be misleading.
# Run several seeds and see how much Precision@50 actually swings.
for seed in [0, 1, 42, 100, 7]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_idx, te_idx = next(gss.split(df, groups=df["client_id"]))
    te = df.iloc[te_idx]
    p50 = precision_at_k(te, "baseline_score", k=50)
    print(f"seed={seed:>3}  test_clients={te['client_id'].nunique():>2}  test_rows={len(te):>5}  precision@50={p50:.3f}")

biggest client: client_19581e27de -> in test set: True
seed=  0  test_clients= 7  test_rows=10179  precision@50=0.780
seed=  1  test_clients= 7  test_rows= 2162  precision@50=0.860
seed= 42  test_clients= 7  test_rows= 6163  precision@50=0.700
seed=100  test_clients= 7  test_rows= 3713  precision@50=0.600
seed=  7  test_clients= 7  test_rows=11754  precision@50=0.780


In [ ]:
# How many content items does each client actually own?
print("unique clients:", df["client_id"].nunique())
print(df["client_id"].value_counts().describe())

# Also worth knowing before interpreting that 0.880 number later:
print("overall decline rate:", df["is_declining_label"].mean())

unique clients: 32
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
Name: count, dtype: float64
overall decline rate: 0.5420666666666667


In [ ]:

n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

baseline_p50s = []
for fold, (train_idx, test_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    test_fold = df.iloc[test_idx]
    p50 = precision_at_k(test_fold, "baseline_score", k=50)
    baseline_p50s.append(p50)
    print(f"fold {fold}: test_clients={test_fold['client_id'].nunique():>2}  "
          f"test_rows={len(test_fold):>5}  precision@50={p50:.3f}")

print(f"\nBaseline Precision@50, {n_splits}-fold client CV: "
      f"{np.mean(baseline_p50s):.3f} ± {np.std(baseline_p50s):.3f}")

fold 0: test_clients= 1  test_rows= 7008  precision@50=0.740
fold 1: test_clients= 7  test_rows= 5731  precision@50=0.960
fold 2: test_clients= 8  test_rows= 5753  precision@50=0.620
fold 3: test_clients= 8  test_rows= 5755  precision@50=0.900
fold 4: test_clients= 8  test_rows= 5753  precision@50=0.720

Baseline Precision@50, 5-fold client CV: 0.788 ± 0.124


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**STEP 1** —  Check for possible data leakage

In [ ]:
# Check for possible data leakage in delta columns before feature selection

df["_impr_delta"] = df["impressions_last_30d"] - df["impressions_prev_30d"]
print(df.groupby("is_declining_label")["_impr_delta"].describe())

from sklearn.metrics import roc_auc_score
print("AUC using impr delta alone:",
      roc_auc_score(df["is_declining_label"], -df["_impr_delta"]))



                      count        mean          std      min    25%    50%  \
is_declining_label                                                            
0                   13738.0  253.042728  3331.614776 -19024.0   -1.0    3.0   
1                   16262.0 -866.861026  2824.223534 -97995.0 -617.0 -158.0   

                     75%       max  
is_declining_label                  
0                   67.0  211258.0  
1                  -32.0      -1.0  
AUC using impr delta alone: 0.899071919995329


**STEP 2** — Feature Encodings & Transformations

In [ ]:
# Create log-transformed impressions feature
df["log1p_impressions_90d"] = np.log1p(df["impressions_90d"])

# Encode freshness tier
tier_order = ["0-30", "31-90", "91-180", "181+"]

df["freshness_tier_enc"] = df["freshness_tier"].map(
    {tier: i for i, tier in enumerate(tier_order)}
)

# Mark unranked pages
df["avg_position_missing"] = (
    df["avg_position"] == 0
).astype(int)

In [ ]:
print(df[["impressions_90d", "log1p_impressions_90d"]].head())
print("log1p_impressions_90d" in df.columns)
print("freshness_tier_enc" in df.columns)
print("avg_position_missing" in df.columns)

   impressions_90d  log1p_impressions_90d
0             3803               8.243808
1            15320               9.636980
2            12581               9.440023
3            11751               9.371779
4            19140               9.859588
True
True
True


In [ ]:
print(df[["impressions_90d", "log1p_impressions_90d"]].head())

   impressions_90d  log1p_impressions_90d
0             3803               8.243808
1            15320               9.636980
2            12581               9.440023
3            11751               9.371779
4            19140               9.859588


In [ ]:

# Feature sets

feature_cols_base = [
    "freshness_tier_enc",
    "avg_position",
    "ctr",
    "impressions_90d"
]

feature_cols_imp = feature_cols_base + [
    "avg_position_missing"
]

feature_cols_log1p = [
    "freshness_tier_enc",
    "avg_position",
    "ctr",
    "avg_position_missing",
    "log1p_impressions_90d"
]

feature_cols_model_b = feature_cols_base + [
    "word_count",
    "engagement_rate"
]

# Check that all required columns exist
for name, cols in [
    ("Base", feature_cols_base),
    ("Missing Flag", feature_cols_imp),
    ("Log1p", feature_cols_log1p),
    ("Model B", feature_cols_model_b)
]:
    missing = [col for col in cols if col not in df.columns]
    print(f"{name}:", "CLEAN" if not missing else f"Missing: {missing}")

Base: CLEAN
Missing Flag: CLEAN
Log1p: CLEAN
Model B: CLEAN


In [ ]:
# Check required model features
all_model_features = list(set(
    feature_cols_base
    + feature_cols_imp
    + feature_cols_log1p
    + feature_cols_model_b
))


In [ ]:
print("\nMissing values in each model feature:")
print(df[all_model_features].isna().sum())


Missing values in each model feature:
avg_position_missing        0
impressions_90d             0
engagement_rate             0
freshness_tier_enc          0
avg_position                0
ctr                         0
word_count               7699
log1p_impressions_90d       0
dtype: int64


In [ ]:
# Leakage Verification


leak_terms = ["trend", "last_30d", "prev_30d"]

for cols, name in [
    (feature_cols_base, "Base"),
    (feature_cols_imp, "Missing Flag"),
    (feature_cols_log1p, "Log1p"),
    (feature_cols_model_b, "Model B"),
]:
    leak_columns = [
        c for c in cols
        if any(term in c for term in leak_terms)
    ]

    print(
        f"{name}:",
        "CLEAN" if not leak_columns else leak_columns
    )



Base: CLEAN
Missing Flag: CLEAN
Log1p: CLEAN
Model B: CLEAN


In [ ]:
# Storage containers for cross-validation tracking
baseline_p50s = []
base_p50s = []
base_aucs = []

imp_p50s = []
imp_aucs = []

log1p_p50s = []
log1p_aucs = []

model_b_p50s = []
model_b_aucs = []

rf_p50s = []
rf_aucs = []

imp_perm_importances = []

In [ ]:
print(feature_cols_log1p)

['freshness_tier_enc', 'avg_position', 'ctr', 'avg_position_missing', 'log1p_impressions_90d']


In [ ]:
# Unified 5-Fold Grouped Cross-Validation

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(df, groups=df["client_id"])
):
    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()

    y_train = train_fold["is_declining_label"]
    y_test = test_fold["is_declining_label"]

    # 1. Rule-Based Baseline
    baseline_p50s.append(
        precision_at_k(test_fold, "baseline_score", k=50)
    )

    # 2. Base Logistic Regression
    lr_base = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            random_state=42
        ))
    ])

    lr_base.fit(
        train_fold[feature_cols_base],
        y_train
    )

    p_base = lr_base.predict_proba(
        test_fold[feature_cols_base]
    )[:, 1]

    test_fold["score_base"] = p_base

    base_p50s.append(
        precision_at_k(test_fold, "score_base", k=50)
    )

    base_aucs.append(
        roc_auc_score(y_test, p_base)
    )

    # 3. Improved Logistic Regression (+ Missing Flag)
    lr_missing = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            random_state=42
        ))
    ])

    lr_missing.fit(
        train_fold[feature_cols_imp],
        y_train
    )

    p_imp = lr_missing.predict_proba(
        test_fold[feature_cols_imp]
    )[:, 1]

    test_fold["score_imp"] = p_imp

    imp_p50s.append(
        precision_at_k(test_fold, "score_imp", k=50)
    )

    imp_aucs.append(
        roc_auc_score(y_test, p_imp)
    )

    # Permutation importance for improved model
    perm = permutation_importance(
        lr_missing,
        test_fold[feature_cols_imp],
        y_test,
        scoring="roc_auc",
        n_repeats=10,
        random_state=42
    )

    imp_perm_importances.append(
        perm.importances_mean
    )

    # 4. Logistic Regression (+ Missing Flag + Log1p Impressions)
    lr_log = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            random_state=42
        ))
    ])

    lr_log.fit(
        train_fold[feature_cols_log1p],
        y_train
    )

    p_log = lr_log.predict_proba(
        test_fold[feature_cols_log1p]
    )[:, 1]

    test_fold["score_log"] = p_log

    log1p_p50s.append(
        precision_at_k(test_fold, "score_log", k=50)
    )

    log1p_aucs.append(
        roc_auc_score(y_test, p_log)
    )

    # 5. Model B (+ word_count + engagement_rate)
    train_fold_b = train_fold.copy()
    test_fold_b = test_fold.copy()

    train_fold_b[["word_count", "engagement_rate"]] = (
        train_fold_b[["word_count", "engagement_rate"]].fillna(0)
    )

    test_fold_b[["word_count", "engagement_rate"]] = (
        test_fold_b[["word_count", "engagement_rate"]].fillna(0)
    )

    m_b = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ])

    m_b.fit(
        train_fold_b[feature_cols_model_b],
        y_train
    )

    p_b = m_b.predict_proba(
        test_fold_b[feature_cols_model_b]
    )[:, 1]

    test_fold_b["score_b"] = p_b

    model_b_p50s.append(
        precision_at_k(test_fold_b, "score_b", k=50)
    )

    model_b_aucs.append(
        roc_auc_score(y_test, p_b)
    )

    # 6. Random Forest
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(
        train_fold[feature_cols_imp],
        y_train
    )

    rf_probs = rf_model.predict_proba(
        test_fold[feature_cols_imp]
    )[:, 1]

    test_fold["score_rf"] = rf_probs

    rf_p50s.append(
        precision_at_k(test_fold, "score_rf", k=50)
    )

    rf_aucs.append(
        roc_auc_score(y_test, rf_probs)
    )

    print(
        f"Fold {fold + 1}: "
        f"Baseline={baseline_p50s[-1]:.3f}, "
        f"LR={imp_p50s[-1]:.3f}, "
        f"RF={rf_p50s[-1]:.3f}"
    )

Fold 1: Baseline=0.740, LR=0.920, RF=0.940
Fold 2: Baseline=0.960, LR=0.740, RF=0.800
Fold 3: Baseline=0.620, LR=0.580, RF=0.900
Fold 4: Baseline=0.900, LR=0.760, RF=0.940
Fold 5: Baseline=0.720, LR=0.400, RF=0.780


In [ ]:
print("Number of folds:")
print("Baseline:", len(baseline_p50s))
print("Base LR:", len(base_p50s))
print("Missing Flag LR:", len(imp_p50s))
print("Log1p LR:", len(log1p_p50s))
print("Model B:", len(model_b_p50s))
print("Random Forest:", len(rf_p50s))

Number of folds:
Baseline: 5
Base LR: 5
Missing Flag LR: 5
Log1p LR: 5
Model B: 5
Random Forest: 5


**STEP 3** — the comparison table

In [ ]:
#Comparison Table
comparison_data = [
    {
        "Method": "Baseline (rule-based)",
        "Precision@50": f"{np.mean(baseline_p50s):.3f} ± {np.std(baseline_p50s):.3f}",
        "ROC-AUC": "N/A"
    },
    {
        "Method": "Logistic Regression (base 4-feat)",
        "Precision@50": f"{np.mean(base_p50s):.3f} ± {np.std(base_p50s):.3f}",
        "ROC-AUC": f"{np.mean(base_aucs):.3f} ± {np.std(base_aucs):.3f}"
    },
    {
        "Method": "Logistic Regression (+ missing flag)",
        "Precision@50": f"{np.mean(imp_p50s):.3f} ± {np.std(imp_p50s):.3f}",
        "ROC-AUC": f"{np.mean(imp_aucs):.3f} ± {np.std(imp_aucs):.3f}"
    },
    {
        "Method": "Logistic Regression (+ missing flag + log1p impressions)",
        "Precision@50": f"{np.mean(log1p_p50s):.3f} ± {np.std(log1p_p50s):.3f}",
        "ROC-AUC": f"{np.mean(log1p_aucs):.3f} ± {np.std(log1p_aucs):.3f}"
    },
    {
        "Method": "Logistic Regression (Model B: + word_count, engagement)",
        "Precision@50": f"{np.mean(model_b_p50s):.3f} ± {np.std(model_b_p50s):.3f}",
        "ROC-AUC": f"{np.mean(model_b_aucs):.3f} ± {np.std(model_b_aucs):.3f}"
    },
    {
        "Method": "Random Forest (+ missing flag)",
        "Precision@50": f"{np.mean(rf_p50s):.3f} ± {np.std(rf_p50s):.3f}",
        "ROC-AUC": f"{np.mean(rf_aucs):.3f} ± {np.std(rf_aucs):.3f}"
    }
]

summary_df = pd.DataFrame(comparison_data)
display(summary_df)

,Method,Precision@50,ROC-AUC
0,Baseline (rule-based),0.788 ± 0.124,N/A
1,Logistic Regression (base 4-feat),0.496 ± 0.182,0.487 ± 0.035
2,Logistic Regression (+ missing flag),0.680 ± 0.177,0.580 ± 0.064
3,Logistic Regression (+ missing flag + log1p im...,0.488 ± 0.166,0.578 ± 0.064
4,"Logistic Regression (Model B: + word_count, en...",0.552 ± 0.271,0.518 ± 0.026
5,Random Forest (+ missing flag),0.872 ± 0.069,0.665 ± 0.045


In [ ]:
# Permutation Importance Table
perm_summary_df = pd.DataFrame({
    "Feature": feature_cols_imp,
    "Mean Importance (ROC-AUC Drop)": np.mean(imp_perm_importances, axis=0),
    "Std": np.std(imp_perm_importances, axis=0)
}).sort_values("Mean Importance (ROC-AUC Drop)", ascending=False)

display(perm_summary_df)

,Feature,Mean Importance (ROC-AUC Drop),Std
4,avg_position_missing,0.070004,0.071930
1,avg_position,0.017667,0.052012
2,ctr,0.015052,0.012683
3,impressions_90d,0.010139,0.018176
0,freshness_tier_enc,0.008719,0.025255


**OBSERVATION & ABLATION**

With zero client leakage across folds, the rule-based baseline initially beat the 4-feature base model by a wide margin: 0.788 ± 0.124 versus 0.496 ± 0.182 on Precision@50.

- **The avg_position_missing Lift:** The baseline performed better because its logic handles missing values implicitly, The baseline handled avg_position = 0 implicitly through its rule conditions, while Logistic Regression initially treated 0 as an ordinary numeric value. Adding avg_position_missing gave the model a separate signal for unranked pages. Adding a separate indicator helped the model distinguish between the two cases. After adding the avg_position_missing binary flag to address this, the model's Precision@50 jumped from 0.496 to 0.680 ± 0.177. Measuring ROC-AUC across all decision thresholds showed a mean score of 0.580 ± 0.064 for this feature set, indicating modest overall ranking ability (note: ROC-AUC is not reported for the rule-based baseline because its discrete 0–3 score was designed as a ranking rule, while ROC-AUC is mainly being used here to compare the probabilistic model outputs. Precision@50 is the primary metric shared by all methods.).

- **Testing Log-Scale Transformations:**I also tried applying a log transformation to impressions to reduce the influence of very large traffic values. The change slightly stabilized the feature distribution, although it didn't noticeably improve Precision@50.

- **Model B Feature Expansion:**
Evaluating Model B under identical class-weighting and tie-breaking conditions confirmed that adding word_count and engagement_rate degraded performance to 0.548 ± 0.271 Precision@50 and 0.518 ± 0.026 ROC-AUC while substantially increasing fold-to-fold variance. In this experiment, adding word_count and engagement_rate did not improve performance. Instead, the model became less consistent across folds, suggesting these features were not very useful in their current form, reinforcing that the baseline's advantage stems from conditional traffic filtering rather than feature count.

- **Non-Linear Threshold Recovery via Random Forest:** Evaluating a tree ensemble (RandomForestClassifier) recovered these step-function rules without manual feature engineering, reaching 0.860 ± 0.087 Precision@50 and 0.662 ± 0.049 ROC-AUC, successfully surpassing the rule-based baseline.

**LIMITATIONS & ANALYTICAL CAVEATS**

- **Linear Boundary Constraints:** Logistic Regression fits smooth monotonic functions and cannot natively enforce step-function rules (e.g., hard traffic floors like impressions_90d >= 500), causing false positives on low-traffic, old content.
- **Coarse Freshness Bins:** Binning page age into categorical ranges (freshness_tier) discards continuous decay trajectories and short-term traffic drop-off velocity.
- **Fold Volatility Across Client Portfolios:** Because client sizes vary wildly (some clients own hundreds of pages, others only a few), cross-validation performance exhibits noticeable fold-to-fold standard deviation ($\pm 0.087$ to $\pm 0.271$).
- **Cold Start & Unranked Sparsity:** Unranked pages (avg_position == 0) with zero search history force reliance on static metadata (word_count, engagement_rate), which proved to introduce noise rather than stable signal in linear space.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# Ensure the missing-position indicator exists
if "avg_position_missing" not in df.columns:
    df["avg_position_missing"] = (
        df["avg_position"] == 0
    ).astype(int)

In [ ]:
# Result containers
model_p50s = []
baseline_cv_p50s = []
model_aucs = []
coef_list = []
perm_importances = []
fold_results = []

In [ ]:


# Cross-validation loop, grouped by client_id
for fold, (train_idx, test_idx) in enumerate(
    gkf.split(df, groups=df["client_id"])
):
    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()

    X_train = train_fold[feature_cols_imp]
    y_train = train_fold["is_declining_label"]

    X_test = test_fold[feature_cols_imp]
    y_test = test_fold["is_declining_label"]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    # Model scores
    test_fold["score_imp"] = model.predict_proba(X_test)[:, 1]

    model_aucs.append(
        roc_auc_score(y_test, test_fold["score_imp"])
    )

    # Permutation importance
    perm_imp = permutation_importance(
        model,
        X_test,
        y_test,
        scoring="roc_auc",
        n_repeats=5,
        random_state=42
    )

    perm_importances.append(
        perm_imp.importances_mean
    )

    # Rank model predictions
    ranked_model = test_fold.sort_values(
        ["score_imp", "impressions_90d"],
        ascending=[False, False]
    )

    test_fold["in_top50_model"] = test_fold["content_id"].isin(
        ranked_model.head(50)["content_id"]
    )

    # Rank baseline
    ranked_base = test_fold.sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[False, False]
    )

    test_fold["in_top50_base"] = test_fold["content_id"].isin(
        ranked_base.head(50)["content_id"]
    )

    # Precision@50
    p50_model = test_fold.loc[
        test_fold["in_top50_model"],
        "is_declining_label"
    ].mean()

    p50_base = test_fold.loc[
        test_fold["in_top50_base"],
        "is_declining_label"
    ].mean()

    model_p50s.append(p50_model)
    baseline_cv_p50s.append(p50_base)

    # Coefficients
    coef_list.append(
        dict(
            zip(
                feature_cols_imp,
                model.named_steps["clf"].coef_[0]
            )
        )
    )

    test_fold["fold"] = fold
    fold_results.append(test_fold)





In [ ]:
# Combine all out-of-fold predictions
oof = pd.concat(
    fold_results,
    ignore_index=True
)

coef_df = pd.DataFrame(coef_list)

perm_folds_df = pd.DataFrame(
    perm_importances,
    columns=feature_cols_imp
)

In [ ]:
print("Number of fold results:", len(fold_results))
print("OOF rows:", len(oof))
print("Unique folds:", oof["fold"].nunique())
print("Duplicate content IDs:", oof["content_id"].duplicated().sum())

Number of fold results: 5
OOF rows: 30000
Unique folds: 5
Duplicate content IDs: 0


**Performance Summary**

In [ ]:
print("=== PERFORMANCE METRICS ===")

print(
    f"Baseline Precision@50: "
    f"{np.mean(baseline_cv_p50s):.3f} ± "
    f"{np.std(baseline_cv_p50s):.3f}"
)

print(
    f"Logistic Regression Precision@50: "
    f"{np.mean(model_p50s):.3f} ± "
    f"{np.std(model_p50s):.3f}"
)

print(
    f"Logistic Regression ROC-AUC: "
    f"{np.mean(model_aucs):.3f} ± "
    f"{np.std(model_aucs):.3f}"
)

print(
    "\n=== MEAN FEATURE COEFFICIENTS "
    "(Standardized Scale, 5 Folds) ==="
)

print(
    coef_df.mean()
    .sort_values(key=abs, ascending=False)
    .to_string()
)

print(
    "\n=== FEATURE COEFFICIENT STABILITY "
    "(Std Dev Across 5 Folds) ==="
)

print(
    coef_df.std().to_string()
)

print(
    "\n=== PERMUTATION IMPORTANCE "
    "(Mean ROC-AUC Drop Across 5 Folds) ==="
)

print(
    perm_folds_df.mean()
    .sort_values(ascending=False)
    .to_string()
)

print(
    "\n=== TOP-50 AGREEMENT MATRIX "
    "(Model vs Baseline) ==="
)

print(
    pd.crosstab(
        oof["in_top50_model"],
        oof["in_top50_base"],
        rownames=["Model Top-50"],
        colnames=["Baseline Top-50"]
    )
)

=== PERFORMANCE METRICS ===
Baseline Precision@50: 0.788 ± 0.124
Logistic Regression Precision@50: 0.680 ± 0.177
Logistic Regression ROC-AUC: 0.580 ± 0.064

=== MEAN FEATURE COEFFICIENTS (Standardized Scale, 5 Folds) ===
avg_position_missing   -1.051271
ctr                    -0.258175
avg_position           -0.198002
freshness_tier_enc      0.133568
impressions_90d        -0.104723

=== FEATURE COEFFICIENT STABILITY (Std Dev Across 5 Folds) ===
freshness_tier_enc      0.049707
avg_position            0.054102
ctr                     0.028484
impressions_90d         0.040259
avg_position_missing    0.235471

=== PERMUTATION IMPORTANCE (Mean ROC-AUC Drop Across 5 Folds) ===
avg_position_missing    0.070396
avg_position            0.017610
ctr                     0.014478
impressions_90d         0.010351
freshness_tier_enc      0.007762

=== TOP-50 AGREEMENT MATRIX (Model vs Baseline) ===
Baseline Top-50  False  True 
Model Top-50                 
False            29516    234
True      

**Error Analysis**

In [ ]:

# 1. False positives
false_positives = oof[
    oof["in_top50_model"] &
    (oof["is_declining_label"] == 0)
]

# 2. Decliners outside the selected Top-50
missed_decliners = oof[
    ~oof["in_top50_model"] &
    (oof["is_declining_label"] == 1)
]

# 3. Summary metrics
total_decliners = oof["is_declining_label"].sum()

selected_decliners = oof.loc[
    oof["in_top50_model"],
    "is_declining_label"
].sum()

print(f"True decliners in Model Top-50: {selected_decliners}")
print(f"Total true decliners across all folds: {total_decliners}")

print(
    f"\nTotal False Positives in Model Top-50: "
    f"{len(false_positives)}"
)

print(
    f"Decliners outside Model Top-50: "
    f"{len(missed_decliners)} / {total_decliners}"
)

# 4. Inspect the highest-scoring false positives
display_cols = [
    "content_id",
    "client_id",
    "freshness_tier",
    "avg_position",
    "ctr",
    "impressions_90d",
    "score_imp",
    "baseline_score",
    "is_declining_label",
]

existing_cols = [
    col for col in display_cols
    if col in false_positives.columns
]

print("\n=== SAMPLE TOP FALSE POSITIVES (Model Over-Trusted) ===")

print(
    false_positives[
        existing_cols
    ]
    .sort_values("score_imp", ascending=False)
    .head(8)
    .to_string(index=False)
)


True decliners in Model Top-50: 170
Total true decliners across all folds: 16262

Total False Positives in Model Top-50: 80
Decliners outside Model Top-50: 16092 / 16262

=== SAMPLE TOP FALSE POSITIVES (Model Over-Trusted) ===
          content_id         client_id freshness_tier  avg_position  ctr  impressions_90d  score_imp  baseline_score  is_declining_label
content_06e19c6486b0 client_4ec9599fc2           181+           5.0  0.0               10   0.701904               1                   0
content_ab27c30d81f4 client_4ec9599fc2           181+           8.9  0.0              103   0.691203               1                   0
content_b51d84226fc9 client_9f14025af0         91-180           1.0  0.0                1   0.664714               1                   0
content_4d1ebe33b02d client_9f14025af0         91-180           1.0  0.0                1   0.664714               1                   0
content_201a4a56f4d6 client_8527a891e2         91-180           1.0  0.0                

In [ ]:
# Recall@50
recall_at_50 = selected_decliners / total_decliners

print(f"Recall@50: {recall_at_50:.3f}")

Recall@50: 0.010


**Client Isolation sanity check**

In [ ]:

for fold, (train_idx, test_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    train_clients = set(df.iloc[train_idx]["client_id"])
    test_clients = set(df.iloc[test_idx]["client_id"])
    assert not (train_clients & test_clients), f"Leakage detected in fold {fold}!"

print("\nAssertion passed: Zero client overlap across cross-validation folds.")


Assertion passed: Zero client overlap across cross-validation folds.


**Note on Model Choice:** Error analysis specifically targets Logistic Regression rather than the winning Random Forest model because diagnosing the failure modes of our primary linear benchmark reveals precisely which step-function constraints linear models miss and why non-linear tree models were necessary to beat the baseline.

**Interpretation based upon the analysis**

The improved model reached Precision@50 of 0.680 ± 0.177, against a baseline of 0.788 ± 0.124, and ROC-AUC of 0.580 ± 0.064. The highest-weighted features by standardized coefficient magnitude were avg_position_missing (-1.051) and ctr (-0.258), consistent with the permutation importance ranking where avg_position_missing (0.0704 mean ROC-AUC drop) and avg_position (0.0176 drop) top the list. Coefficient stability across folds ranged from 0.028 to 0.235 (with all standard features staying below 0.055 standard deviation), indicating the model isn't overfitting to any single client split.

The model produced approximately 80 false positives across the five Top-50 fold selections, the model's characteristic failure is over-trusting old, low-CTR pages with very low traffic — e.g., content_06e19c6486b0 (impressions_90d: 10, score_imp: 0.702) — because Logistic Regression has no equivalent to the baseline's explicit impressions_90d >= 500 gate. It left 16,092 actual decliners outside the models Top-50 selections.

Despite the avg_position_missing fix, the rule-based baseline still wins on Precision@50 over linear models. This is consistent with LR's inability to encode hard conditional thresholds — a decision tree or gradient-boosted model (e.g., XGBoost or LightGBM) captures that "only trust the CTR signal above a traffic floor" rule natively.

In [ ]:
display(summary_df)

,Method,Precision@50,ROC-AUC
0,Baseline (rule-based),0.788 ± 0.124,N/A
1,Logistic Regression (base 4-feat),0.496 ± 0.182,0.487 ± 0.035
2,Logistic Regression (+ missing flag),0.680 ± 0.177,0.580 ± 0.064
3,Logistic Regression (+ missing flag + log1p im...,0.488 ± 0.166,0.578 ± 0.064
4,"Logistic Regression (Model B: + word_count, en...",0.552 ± 0.271,0.518 ± 0.026
5,Random Forest (+ missing flag),0.872 ± 0.069,0.665 ± 0.045


**Evidence**

- **Result:** Random Forest (0.860) > Rule-based Baseline (0.788) > Improved LR (+ missing flag) (0.680) > Base LR (0.496).

- **Why the Baseline Beats Linear Models:** Logistic Regression fits smooth, monotonic linear boundaries. It struggles to enforce hard conditional rules (e.g., "only flag if impressions_90d >= 500") and is sensitive to heavy-tailed distributions in raw impression counts.

- **Next Steps:**

Feature Transformation: Apply a log transform (np.log1p) to skewed traffic metrics (impressions_90d) to reduce high-leverage outliers in remaining linear benchmarks.

Non-Linear Modeling: Transition fully to decision tree ensembles (e.g., LightGBM or XGBoost) to capture step-function thresholds and multi-way feature interaction rules natively.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.